# Music Store Customer Support — Multi-Agent System

Bare-bones but functional customer support workflow using **LangChain**, **LangGraph**, and the **Chinook** SQLite music store database.

**Features**
- Customer verification (ID / email / phone) with human-in-the-loop interrupts
- Music Catalog sub-agent
- Invoice Information sub-agent
- Supervisor routing (music / invoice / both)
- Short-term session memory (`customer_id`) + long-term preference memory

## 1. Setup

Set your Groq API key below (or via a `GROQ_API_KEY` environment variable).

In [ ]:
import os
from getpass import getpass

# Prefer env var; otherwise prompt securely
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

API_KEY = os.environ["GROQ_API_KEY"]
print("API key loaded.")

## 2. Chinook database

Download and load the Chinook SQL script into an in-memory SQLite database.

In [ ]:
from db import get_db, run_query

db = get_db()
print("Tables:", db.get_usable_table_names())
print("Sample customers:")
print(run_query("SELECT CustomerId, FirstName, LastName, Email, Phone FROM Customer LIMIT 5;"))

## 3. Tools (quick smoke test)

In [ ]:
from tools_music import get_albums_by_artist, get_tracks_by_artist
from tools_invoice import get_invoices_by_customer_sorted_by_date

print("Albums by Rolling Stones:")
print(get_albums_by_artist.invoke({"artist": "Rolling Stones"}))
print("\nRecent invoices for customer 1:")
print(get_invoices_by_customer_sorted_by_date.invoke({"customer_id": "1"}))

## 4. Build the multi-agent graph

State schema: `customer_id`, `messages`, `loaded_memory`, `route`.

Flow: **verify → load memory → supervisor → (music | invoice | both) → save memory**

In [ ]:
from memory_store import preference_memory
from graph import ask, build_graph, resume

preference_memory.clear()
graph = build_graph(API_KEY)
print("Graph compiled.")

## 5. Test case 1 — providing customer information

Phone is included, so verification should succeed without an interrupt.
Supervisor should route to **both** (invoice + music).

In [ ]:
THREAD_1 = "testcase-1"

q1 = (
    "My phone number is +55 (12) 3923-5555. "
    "How much was my most recent purchase? "
    "What albums do you have by the Rolling Stones?"
)

result1 = ask(graph, q1, thread_id=THREAD_1)
print("Status:", result1["status"])
print("Customer ID:", result1.get("customer_id"))
print("Memory:", result1.get("loaded_memory"))
print("\nAnswer:\n", result1.get("answer") or result1.get("prompt"))

### Follow-up — preferences from long-term memory

Same session. Preferences should include Rolling Stones from the previous turn.

In [ ]:
q1b = "List some songs that match my preferences?"
result1b = ask(graph, q1b, thread_id=THREAD_1)
print("Status:", result1b["status"])
print("Customer ID:", result1b.get("customer_id"))
print("Memory:", result1b.get("loaded_memory"))
print("\nAnswer:\n", result1b.get("answer") or result1b.get("prompt"))

## 6. Test case 2 — missing customer information (new session)

No credentials → workflow **interrupts** and asks for Customer ID / email / phone.

In [ ]:
THREAD_2 = "testcase-2"

q2 = (
    "How much was my most recent purchase? "
    "What albums do you have by the Rolling Stones?"
)

result2 = ask(graph, q2, thread_id=THREAD_2)
print("Status:", result2["status"])
print(result2.get("prompt") or result2.get("answer"))

### Resume with credentials (human-in-the-loop)

Provide a valid Chinook phone / email / customer id to continue.

In [ ]:
# Example: resume with the same Brazilian phone used in test case 1
result2b = resume(graph, "+55 (12) 3923-5555", thread_id=THREAD_2)
print("Status:", result2b["status"])
print("Customer ID:", result2b.get("customer_id"))
print("\nAnswer:\n", result2b.get("answer") or result2b.get("prompt"))

## 7. Extra checks

In [ ]:
# Music-only (no identity required)
r = ask(graph, "Do you have any Metallica albums?", thread_id="music-only")
print(r.get("answer") or r)

# Show stored preferences
print("\nPreference store:", preference_memory._store)